In [1]:
import pandas as pd 
path = r"C:\Users\papan\Desktop\Coffeomics\sample_information.csv"
metadata = pd.read_csv(path, index_col=0)
metadata = metadata[metadata["Proteomic"]]
metadata = metadata.T
metadata = metadata.iloc[:4,:]


In [2]:
de_results_path = r"C:\Users\papan\Desktop\Coffeomics\proteomics\de_raw_results_50_above10\de_raw_results_de_raw_results.xlsx"
de_results = pd.read_excel(de_results_path, index_col=0)

def parse_contrast_name(name):

    parts = name.split("___")

    attributes = ["Species", "Temperature", "CO2_ppm", "Water_Stress"]

    species = parts[0]
    species = species.replace("small_df_", "")
    species = species.replace("_", "")

    left_vals = parts[1:5]
    right_vals = parts[6:10]

    combined = []

    combined.append(species)   

    for attr, lval, rval in zip(attributes, left_vals, right_vals):

        if lval == rval:
            combined.append(lval)
        else:
            combined.append(f"{lval} vs {rval}")

    return tuple(combined)

# --------------------------------------------------
# Count varying attributes
# --------------------------------------------------
def count_vs_attributes(values):

    return sum(
        " vs " in str(v)
        for v in values
    )

corr = {}

for name in de_results.columns:
    if count_vs_attributes(parse_contrast_name(name))==1:
        corr[name] = parse_contrast_name(name)

len(corr)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\papan\\Desktop\\Coffeomics\\proteomics\\de_raw_results_50_above10\\de_raw_results_de_raw_results.xlsx'

In [36]:
# 1. Keep only selected columns
de_results = de_results.loc[:, de_results.columns.isin(corr)]

# 2. Convert to MultiIndex
import pandas as pd

de_results.columns = pd.MultiIndex.from_tuples(
    [corr[col] for col in de_results.columns],
    names=["Species", "Temperature", "CO2_ppm", "Water_Stress"]
)

# 3. Save
output_path = r"C:\Users\papan\Desktop\Coffeomics\proteomics\de_raw_results_atleast50\de_raw_results_single_factor_only.xlsx"
de_results.to_excel(output_path)

print(f"Kept {len(de_results.columns)} contrasts.")
print(f"Saved filtered file to:\n{output_path}")

Kept 90 contrasts.
Saved filtered file to:
C:\Users\papan\Desktop\Coffeomics\proteomics\de_raw_results_atleast50\de_raw_results_single_factor_only.xlsx


In [1]:
import pandas as pd
de_results_path = r"C:\Users\papan\Desktop\Coffeomics\proteomics\de_raw_results\de_raw_results_50_above10\de_raw_results_de_raw_results.xlsx"
de_results = pd.read_excel(de_results_path, index_col=0, header=0)

In [2]:
results = []

species_list = de_results.columns.get_level_values("Species").unique()

for sp in species_list:
    df_sp = de_results.xs(sp, level="Species", axis=1)

    total_non_nan = df_sp.notna().sum().sum()
    rows_with_any_value = df_sp.notna().any(axis=1).sum()

    results.append({
        "species": sp,
        "non_nan_entries": total_non_nan,
        "rows_with_at_least_one_value": rows_with_any_value
    })

summary = pd.DataFrame(results)
print(summary)

KeyError: 'Requested level (Species) does not match index name (None)'

In [3]:
total_non_nan = de_results.notna().sum().sum()
print(total_non_nan)

69


In [4]:
rows_with_any_value = de_results.notna().any(axis=1).sum()
print(rows_with_any_value)

40


In [57]:
print("Shape:", de_results.shape)
print("Missing values per column:\n", de_results.isna().sum())

Shape: (255, 90)
Missing values per column:
 Species      Temperature                  CO2_ppm           Water_Stress     
C.arabica    25_20°C                      380ppm            medium vs severe     255
                                          380ppm vs 700ppm  medium               245
                                                            severe               255
             25_20°C vs 37_28°C           380ppm            severe               253
             25_20°C vs 42_30°C           380ppm            severe               254
                                                                                ... 
C.canephora  42_30°C vs Rec_14_-_25_20°C  700ppm            without              254
             Rec_14_-_25_20°C             380ppm vs 700ppm  severe               255
                                          380ppm            without vs severe    254
                                          380ppm vs 700ppm  without              255
                           